https://madys.readthedocs.io/en/latest/computation_run.html

* Use madys to confirm the youth using stellar evolutionary model plotted on CMD of <0.1Gyr UBC clusters
* Use other cluster catalogs too

In [ ]:
import madys
getattr(madys, 'MADYS_VERSION', getattr(madys, '__version__', None))

In [ ]:
import sys

basedir = '/ut2/jerome/github/research/project/wakai/'
sys.path.insert(0, f'{basedir}/code/')

In [ ]:
import pandas as pd

ref = "Perren et al. 2023"
#https://zenodo.org/records/8250524
df = pd.read_csv(f'{basedir}/data/Perren2023/UCC_cat.csv')
df.tail()

In [ ]:
df_mem = pd.read_parquet(f'{basedir}/data/Perren2023/UCC_members.parquet')
df_mem.tail()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({"name": "Name",
                        "Source": "GaiaDR3",
                        "probs": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem

In [ ]:
#remove spaces
# df['Name'] = df.Name.apply(lambda x: x.lower().replace(' ', '').split(';')[0])
df.name.isin(df_mem.Name).sum()

In [ ]:
df_mem["source_id"] = df_mem["GaiaDR3"].copy()

In [ ]:
# cname = "hsc2733"
cname = "ubc95"
d = df_mem.query("Name==@cname")
d = d[d.Prob>0.9]
d.shape

In [ ]:
target_name = 'Corot-3'
gaiaDR3id = 4263427117767106816
# nearby_gaiaDR3id = 

In [ ]:
d.query("GaiaDR3==@gaiaDR3id")

In [ ]:
import madys

star_list=[f'Gaia DR3 {gaiaDR3id}']
p=madys.SampleObject(star_list,id_type='DR3',ext_map=None)
p.phot_table

In [ ]:
p.phot_table.to_pandas()

In [ ]:
p.abs_phot

In [ ]:
p.CMD('Gbp-Grp','G','mist')

In [ ]:
import numpy as np

age_old = [20,10,30] #median, min, max
res = p.get_params('mist',
                   mass_range=[0.1,2],
                   age_range=np.array([age_old]),
                   m_unit='m_sun',
                   ph_cut=0.8,
                   n_try=1000)

In [ ]:
res_table = res.to_table()
res_table.round(3)
res_table

In [ ]:
res.pprint('in_notebook')

In [ ]:
age_range = [10,30]
results = []
for model in ['parsec2','mist']:
    r = p.get_params(model, age_range=age_range)
    results.append(r)
ave = madys.FitParams.average_results(results)
ave

In [ ]:
ave.__dict__.keys()

In [ ]:
ave['age'], ave['mass'], ave['radius'], ave['Teff'], ave['logg']

## All members

In [ ]:
star_list2 = d.GaiaDR3.apply(lambda x: f"Gaia DR3 {x}").tolist()
star_list2[:5]

In [ ]:
# because the host is not a cluster member
star_list2.append(f'Gaia DR3 {gaiaDR3id}')

In [ ]:
p2 = madys.SampleObject(star_list2,id_type='DR3',ext_map='leike')
p2.phot_table

In [ ]:
p2.phot_table.to_pandas().shape

In [ ]:
p2.abs_phot.shape

In [ ]:
p2.abs_phot_err.shape

In [ ]:
import numpy as np

abs_phot = np.c_[p2.abs_phot,p2.abs_phot_err]
columns = np.c_[p2.filters,[i+'_err' for i in p2.filters]].T.flatten()
phot = pd.DataFrame(abs_phot, columns=columns)
# phot.to_csv(f'../../data/abs_phot_{cname}_members_from_perren2023.csv', index=False)

In [ ]:
target = pd.DataFrame(p.abs_phot, columns=p.filters).squeeze()
target

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

# model = 'pm13'
# model = 'dartmouth'
# model = 'bt-settl'
# model = 'ames-cond'
# model = 'ames-dusty'
# model = 'nextgen'
model = 'parsec2'
x, y = 'Gbp-Grp', 'G'
plot_ages=[10, 20, 30, 100]
plot_masses=[0.3, 0.5, 0.7, 0.8, 1.0, 1.5, 2]

fig, ax = p2.CMD(
    x, y,
    model,
    xlim=[-0.3, 4.1],
    ylim=[14, -2],
    close=False,
    plot_ages=plot_ages,
    plot_masses=plot_masses,
    label_points=False,
    alpha=0.5,
    s=8,
    figsize=(8, 8)
)

xx = target.Gbp - target.Grp
yy = target.G

ax.plot(
    xx,
    yy,
    marker=r"$\star$",
    mec='k',
    c='r',
    ms=25,
    label=target_name,
    zorder=10,
)

# =========================
# INSET
# =========================

# 1. Define zoom size
dx, dy = 0.4, 0.5

zoom_xlim = [xx - dx, xx + dx + 0.1]
zoom_ylim = [yy + dy, yy - dy]  # magnitude axis (bright at top)

# 2. Create inset axis
axins = ax.inset_axes([0.6, 0.6, 0.35, 0.35])

# 3. Plot content inside inset
madys.IsochroneGrid.plot_isochrones(x, y, 
                                    model, 
                                    plot_ages=plot_ages,
                                    plot_masses=plot_masses,
                                    ax=axins)
madys.SampleObject.plot_photometry(
    x, y, axins, p2,
    errors=None,
    ids=None,
    label_points=False
)

axins.plot(
    xx,
    yy,
    marker=r"$\star$",
    mec='k',
    c='r',
    ms=35,
    zorder=10,
)

# 4. Apply limits (do NOT invert again)
axins.set_xlim(zoom_xlim)
axins.set_ylim(zoom_ylim)

axins.set_facecolor('white')

# 5. Draw connector
# ax.indicate_inset_zoom(axins, edgecolor="black")

# mark_inset(
#     ax, axins,
#     loc1=4, loc2=4,  # try combinations if needed
#     fc="none",
#     ec="black",
#     lw=1
# )

ax.legend(loc='lower left')
ax.set_ylabel(r"$G$")
ax.set_xlabel(r"$G_{BP} - G_{RP}$")
ax.set_title(f"{cname} | {model.upper()} isochrones")
# fig.savefig(f"../../figures/cmd_{model}_{cname}_members_from_perren2023_inset.png", dpi=200)